In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

# =========================================================
# V7
# Goal:
# - Keep the best ideas from v4
# - Add a few physically meaningful features
# - Use a stronger RandomForest
# - Do NOT use Latitude / Longitude as model features
# =========================================================

# =========================
# 1. LOAD TRAINING DATA
# =========================
water_quality = pd.read_csv("../data/water_quality_training_dataset.csv")
landsat = pd.read_csv("../data/landsat_features_training.csv")
terraclimate = pd.read_csv("../data/terraclimate_features_training.csv")

# =========================
# 2. PARSE DATES
# =========================
for df_tmp in [water_quality, landsat, terraclimate]:
    df_tmp["Sample Date"] = pd.to_datetime(df_tmp["Sample Date"], dayfirst=True)

# =========================
# 3. TEMPORAL FEATURES
# =========================
water_quality["month"] = water_quality["Sample Date"].dt.month
water_quality["year"] = water_quality["Sample Date"].dt.year
water_quality["dayofyear"] = water_quality["Sample Date"].dt.dayofyear
water_quality["season"] = ((water_quality["month"] % 12) // 3) + 1

# =========================
# 4. MERGE TRAINING DATA
# =========================
df = water_quality.merge(
    landsat,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

df = df.merge(
    terraclimate,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

# =========================
# 5. FEATURE ENGINEERING
# =========================
# Base features from previous best versions
df["nir_swir16_ratio"] = df["nir"] / df["swir16"]
df["nir_swir22_ratio"] = df["nir"] / df["swir22"]
df["green_nir_ratio"] = df["green"] / df["nir"]

df["nir_minus_swir16"] = df["nir"] - df["swir16"]
df["nir_minus_green"] = df["nir"] - df["green"]

df["ndmi_pet"] = df["NDMI"] * df["pet"]
df["mndwi_pet"] = df["MNDWI"] * df["pet"]

df["swir_ratio"] = df["swir16"] / df["swir22"]

df["nir_pet"] = df["nir"] * df["pet"]
df["swir16_pet"] = df["swir16"] * df["pet"]
df["ndmi_day"] = df["NDMI"] * df["dayofyear"]

df["nir_green_ratio"] = df["nir"] / df["green"]
df["swir16_green_ratio"] = df["swir16"] / df["green"]
df["pet_year"] = df["pet"] * df["year"]
df["ndmi_pet_day"] = df["NDMI"] * df["pet"] * df["dayofyear"]
df["mndwi_pet_day"] = df["MNDWI"] * df["pet"] * df["dayofyear"]

# New v7 features
df["nir_swir_diff"] = df["nir"] - df["swir22"]
df["water_index"] = (df["green"] - df["swir16"]) / (df["green"] + df["swir16"])
df["vegetation_water_interaction"] = df["NDMI"] * df["MNDWI"]
df["pet_month"] = df["pet"] * df["month"]
df["pet_season"] = df["pet"] * df["season"]

# =========================
# 6. HANDLE MISSING VALUES
# =========================
train_medians = df.median(numeric_only=True)
df = df.fillna(train_medians)

# =========================
# 7. DEFINE X / y
# =========================
targets = [
    "Total Alkalinity",
    "Electrical Conductance",
    "Dissolved Reactive Phosphorus"
]

X = df.drop(columns=targets + ["Sample Date", "Latitude", "Longitude"])
y = df[targets]

# =========================
# 8. TRAIN MODEL
# =========================
rf_final = RandomForestRegressor(
    n_estimators=1500,
    max_depth=14,
    min_samples_leaf=1,
    max_features="sqrt",
    random_state=42,
    n_jobs=-1
)

rf_final.fit(X, y)

# =========================
# 9. LOAD SUBMISSION DATA
# =========================
submission = pd.read_csv("../data/submission_template.csv")
landsat_val = pd.read_csv("../data/landsat_features_validation.csv")
terraclimate_val = pd.read_csv("../data/terraclimate_features_validation.csv")

# =========================
# 10. PARSE DATES
# =========================
for df_tmp in [submission, landsat_val, terraclimate_val]:
    df_tmp["Sample Date"] = pd.to_datetime(df_tmp["Sample Date"], dayfirst=True)

# =========================
# 11. TEMPORAL FEATURES
# =========================
submission["month"] = submission["Sample Date"].dt.month
submission["year"] = submission["Sample Date"].dt.year
submission["dayofyear"] = submission["Sample Date"].dt.dayofyear
submission["season"] = ((submission["month"] % 12) // 3) + 1

# =========================
# 12. MERGE VALIDATION DATA
# =========================
df_val = submission.merge(
    landsat_val,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

df_val = df_val.merge(
    terraclimate_val,
    on=["Latitude", "Longitude", "Sample Date"],
    how="left"
)

# =========================
# 13. SAME FEATURE ENGINEERING
# =========================
df_val["nir_swir16_ratio"] = df_val["nir"] / df_val["swir16"]
df_val["nir_swir22_ratio"] = df_val["nir"] / df_val["swir22"]
df_val["green_nir_ratio"] = df_val["green"] / df_val["nir"]

df_val["nir_minus_swir16"] = df_val["nir"] - df_val["swir16"]
df_val["nir_minus_green"] = df_val["nir"] - df_val["green"]

df_val["ndmi_pet"] = df_val["NDMI"] * df_val["pet"]
df_val["mndwi_pet"] = df_val["MNDWI"] * df_val["pet"]

df_val["swir_ratio"] = df_val["swir16"] / df_val["swir22"]

df_val["nir_pet"] = df_val["nir"] * df_val["pet"]
df_val["swir16_pet"] = df_val["swir16"] * df_val["pet"]
df_val["ndmi_day"] = df_val["NDMI"] * df_val["dayofyear"]

df_val["nir_green_ratio"] = df_val["nir"] / df_val["green"]
df_val["swir16_green_ratio"] = df_val["swir16"] / df_val["green"]
df_val["pet_year"] = df_val["pet"] * df_val["year"]
df_val["ndmi_pet_day"] = df_val["NDMI"] * df_val["pet"] * df_val["dayofyear"]
df_val["mndwi_pet_day"] = df_val["MNDWI"] * df_val["pet"] * df_val["dayofyear"]

# New v7 features
df_val["nir_swir_diff"] = df_val["nir"] - df_val["swir22"]
df_val["water_index"] = (df_val["green"] - df_val["swir16"]) / (df_val["green"] + df_val["swir16"])
df_val["vegetation_water_interaction"] = df_val["NDMI"] * df_val["MNDWI"]
df_val["pet_month"] = df_val["pet"] * df_val["month"]
df_val["pet_season"] = df_val["pet"] * df_val["season"]

# =========================
# 14. HANDLE MISSING VALUES
# =========================
df_val = df_val.fillna(train_medians)

# =========================
# 15. PREPARE VALIDATION FEATURES
# =========================
X_val = df_val.drop(
    columns=[
        "Sample Date",
        "Latitude",
        "Longitude",
        "Total Alkalinity",
        "Electrical Conductance",
        "Dissolved Reactive Phosphorus"
    ],
    errors="ignore"
)

X_val = X_val[X.columns]

# =========================
# 16. PREDICT
# =========================
predictions = rf_final.predict(X_val)

# =========================
# 17. BUILD SUBMISSION
# =========================
submission["Total Alkalinity"] = predictions[:, 0]
submission["Electrical Conductance"] = predictions[:, 1]
submission["Dissolved Reactive Phosphorus"] = predictions[:, 2]

submission_v7 = submission[
    [
        "Longitude",
        "Latitude",
        "Sample Date",
        "Total Alkalinity",
        "Electrical Conductance",
        "Dissolved Reactive Phosphorus"
    ]
]

# =========================
# 18. EXPORT
# =========================
submission_v7.to_csv("../submissions/submission_v7.csv", index=False)

# =========================
# 19. QUICK CHECK
# =========================
print(submission_v7.shape)
print(submission_v7.head())

(200, 6)
   Longitude   Latitude Sample Date  Total Alkalinity  Electrical Conductance  \
0  27.822778 -32.043333  2014-09-01        115.957047              426.977981   
1  26.077500 -33.329167  2015-09-16        139.107335              547.216542   
2  27.640028 -32.991639  2015-05-07         73.933814              423.557247   
3  24.439167 -34.096389  2012-02-07         49.703216              141.227325   
4  28.581667 -32.000556  2014-10-01        104.101545              410.863403   

   Dissolved Reactive Phosphorus  
0                      35.274142  
1                      50.050961  
2                      30.952932  
3                      14.721298  
4                      24.905110  
